# RQ4, Part 1: Real Analysis —OLS Sensitivity + Cox Survival

Real analysis of the SEC EDGAR material-weakness remediation dataset (N=113, individually verified).

### Methodological hierarchy
- **Primary analysis:** Cox proportional hazards model using all 113 observations and correctly handling the 52 right-censored cases.
- **Sensitivity analysis:** OLS regression restricted to the 61 remediated cases, because remediation duration is directly observed only when remediation occurred.
- **Additional sensitivity:** Cox model without disclosure year.
- **Diagnostic:** proportional-hazards assumption check.

**Requires:** `rq4_real_sec_edgar_dataset_FINAL.csv`

In [1]:
!pip install -q pandas numpy statsmodels lifelines scipy || pip install -q pandas numpy statsmodels lifelines scipy --break-system-packages

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.3 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

df = pd.read_csv(
    'rq4_real_sec_edgar_dataset_FINAL.csv',
    parse_dates=['disclosure_date', 'remediation_date']
)

print(f'Loaded real RQ4 dataset: N = {len(df)} real company material-weakness episodes')
print(f"Remediated: {int(df['event_observed'].sum())}, Still open (censored): {int((df['event_observed'] == 0).sum())}")

Loaded real RQ4 dataset: N = 113 real company material-weakness episodes
Remediated: 61, Still open (censored): 52


In [3]:
# ---------- Validate event and duration fields ----------
df['event_observed'] = pd.to_numeric(df['event_observed'], errors='coerce')
df['remediation_days'] = pd.to_numeric(df['remediation_days'], errors='coerce')

assert df['event_observed'].notna().all(), 'event_observed contains missing values.'
assert set(df['event_observed'].unique()).issubset({0, 1}), 'event_observed must contain only 0/1.'
assert (df['remediation_days'] >= 0).all(), 'Negative remediation_days detected.'

n_total = len(df)
n_remediated = int((df['event_observed'] == 1).sum())
n_censored = int((df['event_observed'] == 0).sum())

print('=' * 70)
print('RQ4 EVENT / CENSORING STRUCTURE')
print('=' * 70)
print(f'Total observations : {n_total:,}')
print(f'Remediated/events  : {n_remediated:,}')
print(f'Right-censored     : {n_censored:,}')
print(f'Remediated         : {n_remediated / n_total * 100:.2f}%')
print(f'Censored           : {n_censored / n_total * 100:.2f}%')

assert n_remediated + n_censored == n_total

RQ4 EVENT / CENSORING STRUCTURE
Total observations : 113
Remediated/events  : 61
Right-censored     : 52
Remediated         : 53.98%
Censored           : 46.02%


In [4]:
# ---------- Consolidate high-cardinality categories (necessary at N=113) ----------
def consolidate_weakness(cat):
    cat = str(cat)
    if 'Revenue' in cat:
        return 'Revenue Recognition'
    if 'ITGC' in cat:
        return 'ITGC'
    if 'Complex' in cat or 'Warrant' in cat or 'Instrument' in cat:
        return 'Complex Transactions/Instruments'
    if 'Control Environment' in cat or 'Risk Assessment' in cat or 'Staffing' in cat or 'Segregation' in cat:
        return 'Control Environment/Staffing'
    return 'Other'

def consolidate_industry(ind):
    ind = str(ind)
    if 'Technology' in ind:
        return 'Technology'
    if 'Biotech' in ind or 'Healthcare' in ind:
        return 'Biotech/Healthcare'
    if 'Manufacturing' in ind or 'Industrial' in ind or 'Aerospace' in ind or 'Mining' in ind:
        return 'Manufacturing/Industrial'
    if 'SPAC' in ind:
        return 'SPAC'
    if 'Energy' in ind:
        return 'Energy'
    if 'Financial' in ind or 'Insurance' in ind or 'Real Estate' in ind:
        return 'Financial/Real Estate'
    return 'Media/Consumer/Other'

df['weakness_group'] = df['weakness_category'].apply(consolidate_weakness)
df['industry_group'] = df['industry'].apply(consolidate_industry)

print('Consolidated weakness_group distribution:')
print(df['weakness_group'].value_counts())

print('\nConsolidated industry_group distribution:')
print(df['industry_group'].value_counts())


Consolidated weakness_group distribution:
weakness_group
Other                               53
Control Environment/Staffing        20
Revenue Recognition                 15
ITGC                                13
Complex Transactions/Instruments    12
Name: count, dtype: int64

Consolidated industry_group distribution:
industry_group
Biotech/Healthcare          27
Technology                  23
Media/Consumer/Other        20
Manufacturing/Industrial    19
Financial/Real Estate       14
Energy                       6
SPAC                         4
Name: count, dtype: int64


In [5]:
# ---------- Disclosure year and centered disclosure year ----------
df['disclosure_year'] = df['disclosure_date'].dt.year
df['disclosure_year_c'] = df['disclosure_year'] - df['disclosure_year'].mean()

print(f"Disclosure year range: {int(df['disclosure_year'].min())} to {int(df['disclosure_year'].max())}")
print(f"Mean disclosure year: {df['disclosure_year'].mean():.2f}")
print(f"Centered year range: {df['disclosure_year_c'].min():.2f} to {df['disclosure_year_c'].max():.2f}")

Disclosure year range: 2017 to 2024
Mean disclosure year: 2020.30
Centered year range: -3.30 to 3.70


## RQ4a — OLS sensitivity analysis

OLS is restricted to the 61 remediated cases. The 52 right-censored cases are excluded because their observed duration represents follow-up time up to censoring rather than completed remediation time. The Cox model below is the primary analysis because it uses all 113 observations and explicitly handles censoring.

In [6]:
# ---------- RQ4a: OLS sensitivity analysis ----------
ols_remediated = df[df['event_observed'] == 1].copy()

print('=' * 70)
print('RQ4a: OLS SENSITIVITY ANALYSIS')
print('=' * 70)
print(f'Remediated cases used for OLS: {len(ols_remediated):,}')
print(f'Right-censored cases excluded from OLS: {(df["event_observed"] == 0).sum():,}')

reg_model = smf.ols(
    'remediation_days ~ C(weakness_group) + C(industry_group) + disclosure_year_c',
    data=ols_remediated
).fit()

print(reg_model.summary())

RQ4a: OLS SENSITIVITY ANALYSIS
Remediated cases used for OLS: 61
Right-censored cases excluded from OLS: 52
                            OLS Regression Results                            
Dep. Variable:       remediation_days   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     1.115
Date:                Fri, 21 Aug 2026   Prob (F-statistic):              0.370
Time:                        07:02:16   Log-Likelihood:                -453.12
No. Observations:                  61   AIC:                             930.2
Df Residuals:                      49   BIC:                             955.6
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.

In [7]:
# ---------- OLS key results ----------
print('=' * 70)
print('OLS KEY RESULTS')
print('=' * 70)
print(f'Observations: {int(reg_model.nobs):,}')
print(f'R-squared: {reg_model.rsquared:.4f}')
print(f'Adjusted R-squared: {reg_model.rsquared_adj:.4f}')
print(f'F-test p-value: {reg_model.f_pvalue:.4f}')

OLS KEY RESULTS
Observations: 61
R-squared: 0.2002
Adjusted R-squared: 0.0206
F-test p-value: 0.3705


## Caveat: sample size vs. model complexity

At N=113 (61 events, 52 right-censored), the Cox model above includes ~11 estimated
parameters (4 weakness-group dummies + 6 industry-group dummies + centered
disclosure year), giving an events-per-variable ratio of roughly 5.5. This is below
the conventional EPV >= 10 rule of thumb for Cox regression, and increases the risk
of unstable or overfit coefficient estimates.

The OLS sensitivity model shows the same strain directly: R-squared = 0.200 but
adjusted R-squared = 0.021, with an overall F-test p = 0.370 (not significant). The
gap between R-squared and adjusted R-squared at N=61 with 11 predictors is a classic
sign that the model is absorbing noise from the categorical dummies rather than
detecting real structure.

This should be stated plainly in the interim report as a limitation of RQ4's current
sample size, not just implied by the R-squared/adjusted-R-squared numbers in the
output table.


## RQ4b — Primary Cox proportional hazards model

The Cox model is the primary inferential model because it uses all observations and correctly handles the 52 right-censored cases. Predictors are weakness group, industry group, and centered disclosure year.

In [8]:
# ---------- RQ4b: Cox proportional hazards model ----------
surv_df = df[
    ['remediation_days', 'event_observed', 'weakness_group', 'industry_group', 'disclosure_year_c']
].copy()

surv_df = pd.get_dummies(
    surv_df,
    columns=['weakness_group', 'industry_group'],
    drop_first=True
)

for col in surv_df.columns:
    if surv_df[col].dtype == bool:
        surv_df[col] = surv_df[col].astype(float)

for col in ['remediation_days', 'event_observed', 'disclosure_year_c']:
    surv_df[col] = pd.to_numeric(surv_df[col], errors='coerce')

surv_df = surv_df.dropna().copy()

print('=' * 70)
print('RQ4b: PRIMARY COX PROPORTIONAL HAZARDS MODEL')
print('=' * 70)
print(f'Observations: {len(surv_df):,}')
print(f'Observed events: {int(surv_df["event_observed"].sum()):,}')
print(f'Right-censored: {int((surv_df["event_observed"] == 0).sum()):,}')

cph = CoxPHFitter()
cph.fit(
    surv_df,
    duration_col='remediation_days',
    event_col='event_observed'
)

cph.print_summary()
print(f'\nConcordance index (C-statistic): {cph.concordance_index_:.4f}')

RQ4b: PRIMARY COX PROPORTIONAL HAZARDS MODEL
Observations: 113
Observed events: 61
Right-censored: 52


<lifelines.CoxPHFitter: fitted with 113 total observations, 52 right-censored observations>
             duration col = 'remediation_days'
                event col = 'event_observed'
      baseline estimation = breslow
   number of observations = 113
number of events observed = 61
   partial log-likelihood = -261.53
         time fit was run = 2026-08-21 07:02:17 UTC

---
                                             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                    
disclosure_year_c                           -0.09      0.91      0.09           -0.27            0.10                0.76                1.10
weakness_group_Control Environment/Staffing -0.32      0.73      0.62           -1.53            0.89                0.22                2.44
weakness_group_ITGC                          0.09      1.09      0.63           -1.14            1.32                0.32                3.73
weakness_group_Other                         0.07      1.07      0.54           -0.98            1.12                0.37                3.07
weakness_group_Revenue Recognition          -0.13      0.88      0.59           -1.29            1.04                0.27                2.82
industry_group_Energy                        0.86      2.36      0.65           -0.42            2.14                0.66                8.49
industry_group_Financial/Real Estate         0.32      1.37      0.49           -0.63            1.27                0.53                3.56
industry_group_Manufacturing/Industrial      0.50      1.65      0.45           -0.37            1.37                0.69                3.95
industry_group_Media/Consumer/Other          0.64      1.89      0.43           -0.20            1.48                0.82                4.38
industry_group_SPAC                          0.52      1.68      0.93           -1.30            2.34                0.27               10.38
industry_group_Technology                    0.53      1.71      0.42           -0.28            1.35                0.75                3.86

                                             cmp to     z    p  -log2(p)
covariate                                                               
disclosure_year_c                              0.00 -0.95 0.34      1.55
weakness_group_Control Environment/Staffing    0.00 -0.52 0.60      0.73
weakness_group_ITGC                            0.00  0.14 0.89      0.17
weakness_group_Other                           0.00  0.13 0.89      0.16
weakness_group_Revenue Recognition             0.00 -0.21 0.83      0.27
industry_group_Energy                          0.00  1.32 0.19      2.42
industry_group_Financial/Real Estate           0.00  0.65 0.51      0.96
industry_group_Manufacturing/Industrial        0.00  1.12 0.26      1.93
industry_group_Media/Consumer/Other            0.00  1.50 0.13      2.89
industry_group_SPAC                            0.00  0.56 0.57      0.80
industry_group_Technology                      0.00  1.28 0.20      2.33
---
Concordance = 0.62
Partial AIC = 545.06
log-likelihood ratio test = 6.44 on 11 df
-log2(p) of ll-ratio test = 0.25


Concordance index (C-statistic): 0.6199


In [9]:
# ---------- Cox hazard ratios and 95% confidence intervals ----------
cox_results = cph.summary[
    [
        'coef', 'exp(coef)', 'se(coef)', 'p',
        'exp(coef) lower 95%', 'exp(coef) upper 95%'
    ]
].copy()

cox_results.columns = [
    'Coefficient', 'Hazard Ratio', 'SE', 'p-value',
    'HR Lower 95%', 'HR Upper 95%'
]

display(cox_results.round(4))

,Coefficient,Hazard Ratio,SE,p-value,HR Lower 95%,HR Upper 95%
covariate,,,,,,
disclosure_year_c,-0.0896,0.9143,0.0943,0.3420,0.7601,1.0999
weakness_group_Control Environment/Staffing,-0.3207,0.7257,0.6184,0.6041,0.2159,2.4386
weakness_group_ITGC,0.0862,1.0900,0.6271,0.8907,0.3189,3.7259
weakness_group_Other,0.0709,1.0735,0.5369,0.8949,0.3748,3.0749
weakness_group_Revenue Recognition,-0.1277,0.8801,0.5946,0.8299,0.2744,2.8226
industry_group_Energy,0.8604,2.3640,0.6526,0.1874,0.6579,8.4942
industry_group_Financial/Real Estate,0.3176,1.3739,0.4858,0.5132,0.5302,3.5604
industry_group_Manufacturing/Industrial,0.5002,1.6490,0.4460,0.2620,0.6880,3.9523
industry_group_Media/Consumer/Other,0.6392,1.8949,0.4274,0.1348,0.8200,4.3788


In [10]:
# ---------- RQ4c: proportional hazards assumption check ----------
print('=' * 70)
print('RQ4c: PROPORTIONAL HAZARDS ASSUMPTION CHECK')
print('=' * 70)

try:
    cph.check_assumptions(
        surv_df,
        p_value_threshold=0.05,
        show_plots=False
    )
except Exception as e:
    print('The proportional-hazards diagnostic could not be completed.')
    print(f'Diagnostic message: {e}')

RQ4c: PROPORTIONAL HAZARDS ASSUMPTION CHECK
The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'disclosure_year_c' failed the non-proportional test: p-value is 0.0337.

   Advice 1: the functional form of the variable 'disclosure_year_c' might be incorrect. That is,
there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'disclosure_year_c' using pd.cut, and then specify it in
`strata=['disclosure_year_c', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


---
[A]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html
[B]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Bin-variable-and-stratify-on-it
[C]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Prop

## RQ4d — Cox model with binned/stratified disclosure year (proportional-hazards fix)

The diagnostic above showed `disclosure_year_c` failing the proportional-hazards
test (p = 0.034). Simply dropping the variable (the sensitivity model further
below) sidesteps the diagnostic rather than resolving it, and discards a
predictor the OLS and primary Cox models both retained.

Per `lifelines`' own guidance, the fix used here is to **bin disclosure year
into strata** and refit with `strata=['year_stratum']`, which allows the
baseline hazard to differ freely across year bins rather than assuming a
single proportional effect across the full 2017-2024 range. Bins are chosen
to give roughly balanced group sizes given N=113.

In [11]:
# ---------- RQ4d: PH-assumption fix via binned/stratified disclosure year ----------
bins = [2016, 2019, 2021, 2025]
labels = ['2017-2019', '2020-2021', '2022-2024']
df['year_stratum'] = pd.cut(df['disclosure_year'], bins=bins, labels=labels)

print('Year stratum counts:')
print(df['year_stratum'].value_counts().sort_index())
print('\nEvents per stratum:')
print(df.groupby('year_stratum')['event_observed'].sum())

surv_df_strat = df[
    ['remediation_days', 'event_observed', 'weakness_group', 'industry_group', 'year_stratum']
].copy()

surv_df_strat = pd.get_dummies(
    surv_df_strat,
    columns=['weakness_group', 'industry_group'],
    drop_first=True
)

for col in surv_df_strat.columns:
    if surv_df_strat[col].dtype == bool:
        surv_df_strat[col] = surv_df_strat[col].astype(float)

for col in ['remediation_days', 'event_observed']:
    surv_df_strat[col] = pd.to_numeric(surv_df_strat[col], errors='coerce')

surv_df_strat = surv_df_strat.dropna().copy()

cph_strat = CoxPHFitter()
cph_strat.fit(
    surv_df_strat,
    duration_col='remediation_days',
    event_col='event_observed',
    strata=['year_stratum']
)

print('=' * 70)
print('RQ4d: STRATIFIED COX MODEL (proportional-hazards fix)')
print('=' * 70)
cph_strat.print_summary()
print(f'\nConcordance index (stratified model): {cph_strat.concordance_index_:.4f}')

print('\n--- Re-checking proportional-hazards assumption on the stratified model ---')
try:
    cph_strat.check_assumptions(surv_df_strat, p_value_threshold=0.05, show_plots=False)
except Exception as e:
    print('Diagnostic message:', e)

Year stratum counts:
year_stratum
2017-2019    48
2020-2021    39
2022-2024    26
Name: count, dtype: int64

Events per stratum:
year_stratum
2017-2019    31
2020-2021    16
2022-2024    14
Name: event_observed, dtype: int64
RQ4d: STRATIFIED COX MODEL (proportional-hazards fix)


<lifelines.CoxPHFitter: fitted with 113 total observations, 52 right-censored observations>
             duration col = 'remediation_days'
                event col = 'event_observed'
                   strata = year_stratum
      baseline estimation = breslow
   number of observations = 113
number of events observed = 61
   partial log-likelihood = -198.49
         time fit was run = 2026-08-21 07:02:17 UTC

---
                                             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                    
weakness_group_Control Environment/Staffing -0.40      0.67      0.62           -1.62            0.81                0.20                2.25
weakness_group_ITGC                          0.10      1.11      0.62           -1.12            1.33                0.33                3.76
weakness_group_Other                        -0.06      0.94      0.54           -1.12            1.01                0.33                2.73
weakness_group_Revenue Recognition          -0.24      0.78      0.61           -1.43            0.95                0.24                2.58
industry_group_Energy                        1.14      3.14      0.66           -0.14            2.43                0.87               11.38
industry_group_Financial/Real Estate         0.19      1.21      0.49           -0.76            1.15                0.47                3.15
industry_group_Manufacturing/Industrial      0.49      1.64      0.44           -0.38            1.36                0.68                3.92
industry_group_Media/Consumer/Other          0.78      2.18      0.44           -0.07            1.64                0.93                5.13
industry_group_SPAC                          0.97      2.63      0.97           -0.93            2.87                0.39               17.57
industry_group_Technology                    0.52      1.68      0.42           -0.30            1.34                0.74                3.81

                                             cmp to     z    p  -log2(p)
covariate                                                               
weakness_group_Control Environment/Staffing    0.00 -0.65 0.52      0.96
weakness_group_ITGC                            0.00  0.17 0.87      0.20
weakness_group_Other                           0.00 -0.11 0.91      0.13
weakness_group_Revenue Recognition             0.00 -0.40 0.69      0.54
industry_group_Energy                          0.00  1.74 0.08      3.62
industry_group_Financial/Real Estate           0.00  0.40 0.69      0.53
industry_group_Manufacturing/Industrial        0.00  1.11 0.27      1.90
industry_group_Media/Consumer/Other            0.00  1.79 0.07      3.77
industry_group_SPAC                            0.00  1.00 0.32      1.65
industry_group_Technology                      0.00  1.25 0.21      2.24
---
Concordance = 0.61
Partial AIC = 416.99
log-likelihood ratio test = 7.41 on 10 df
-log2(p) of ll-ratio test = 0.54


Concordance index (stratified model): 0.6138

--- Re-checking proportional-hazards assumption on the stratified model ---
Proportional hazard assumption looks okay.


## RQ4e — Cox sensitivity without disclosure year

This is a secondary sensitivity check, separate from the RQ4d fix above. It
assesses whether model discrimination materially depends on disclosure year at
all, by removing it entirely, rather than fixing its functional form. RQ4d's
stratified model above is the preferred way to retain disclosure year while
resolving the proportional-hazards violation; this no-year model is kept only
as an additional sensitivity check, not as the primary response to the PH
diagnostic.

In [12]:
# ---------- Cox sensitivity model without disclosure year ----------
surv_df_no_year = df[
    ['remediation_days', 'event_observed', 'weakness_group', 'industry_group']
].copy()

surv_df_no_year = pd.get_dummies(
    surv_df_no_year,
    columns=['weakness_group', 'industry_group'],
    drop_first=True
)

for col in surv_df_no_year.columns:
    if surv_df_no_year[col].dtype == bool:
        surv_df_no_year[col] = surv_df_no_year[col].astype(float)

for col in ['remediation_days', 'event_observed']:
    surv_df_no_year[col] = pd.to_numeric(surv_df_no_year[col], errors='coerce')

surv_df_no_year = surv_df_no_year.dropna().copy()

cph_no_year = CoxPHFitter()
cph_no_year.fit(
    surv_df_no_year,
    duration_col='remediation_days',
    event_col='event_observed'
)

print('=' * 70)
print('COX DISCLOSURE-YEAR SENSITIVITY')
print('=' * 70)
print(f'C-index with disclosure year: {cph.concordance_index_:.4f}')
print(f'C-index without disclosure year: {cph_no_year.concordance_index_:.4f}')

display(
    cph_no_year.summary[
        ['coef', 'exp(coef)', 'p', 'exp(coef) lower 95%', 'exp(coef) upper 95%']
    ].round(4)
)

COX DISCLOSURE-YEAR SENSITIVITY
C-index with disclosure year: 0.6199
C-index without disclosure year: 0.6059


,coef,exp(coef),p,exp(coef) lower 95%,exp(coef) upper 95%
covariate,,,,,
weakness_group_Control Environment/Staffing,-0.2821,0.7542,0.6464,0.2259,2.5173
weakness_group_ITGC,0.0127,1.0128,0.9836,0.3016,3.4013
weakness_group_Other,0.0838,1.0874,0.8752,0.3819,3.0967
weakness_group_Revenue Recognition,-0.0419,0.9589,0.9426,0.3066,2.9991
industry_group_Energy,0.8261,2.2845,0.2039,0.6386,8.1720
industry_group_Financial/Real Estate,0.3005,1.3505,0.5362,0.5213,3.4988
industry_group_Manufacturing/Industrial,0.6000,1.8221,0.1660,0.7797,4.2583
industry_group_Media/Consumer/Other,0.7032,2.0201,0.0940,0.8871,4.6001
industry_group_SPAC,0.4962,1.6425,0.5915,0.2682,10.0604


In [13]:
# ---------- Final RQ4 evidence matrix ----------
ols_f_p = reg_model.f_pvalue
ols_r2 = reg_model.rsquared
cox_cindex = cph.concordance_index_
cox_strat_cindex = cph_strat.concordance_index_
cox_no_year_cindex = cph_no_year.concordance_index_

evidence_rows = [
    {
        'Evidence': 'OLS sensitivity — remediated cases only',
        'Estimate': ols_r2,
        'Metric': 'R²',
        'p-value': ols_f_p,
        'Supported?': ols_f_p < 0.05,
        'Interpretation': (
            'Statistically significant evidence that the specified predictors explain variation in observed remediation duration.'
            if ols_f_p < 0.05 else
            'No statistically significant evidence that weakness group, industry group, and disclosure year explain observed remediation duration.'
        )
    },
    {
        'Evidence': 'Primary Cox proportional hazards model (unstratified)',
        'Estimate': cox_cindex,
        'Metric': 'C-index',
        'p-value': np.nan,
        'Supported?': np.nan,
        'Interpretation': 'Time-to-remediation model incorporating observed events and right-censored cases. Fails the proportional-hazards check for disclosure_year_c (p=0.034).'
    },
    {
        'Evidence': 'RQ4d — Cox model, stratified by year (PH-assumption fix)',
        'Estimate': cox_strat_cindex,
        'Metric': 'C-index',
        'p-value': np.nan,
        'Supported?': np.nan,
        'Interpretation': 'Preferred model: retains disclosure year via stratified baseline hazard, resolving the proportional-hazards violation rather than dropping the variable.'
    },
    {
        'Evidence': 'RQ4e — Cox model, disclosure-year removed (sensitivity only)',
        'Estimate': cox_no_year_cindex,
        'Metric': 'C-index',
        'p-value': np.nan,
        'Supported?': np.nan,
        'Interpretation': 'Secondary sensitivity check assessing whether model discrimination materially depends on disclosure year. Not the primary response to the PH diagnostic — see RQ4d.'
    }
]

evidence_matrix = pd.DataFrame(evidence_rows)
evidence_display = evidence_matrix.copy()
evidence_display['p-value'] = evidence_display['p-value'].apply(
    lambda x: '< 0.001' if pd.notna(x) and x < 0.001 else (f'{x:.3f}' if pd.notna(x) else '—')
)

display(evidence_display.round(4))

,Evidence,Estimate,Metric,p-value,Supported?,Interpretation
0,OLS sensitivity — remediated cases only,0.2002,R²,0.370,False,No statistically significant evidence that wea...
1,Primary Cox proportional hazards model (unstra...,0.6199,C-index,—,NaN,Time-to-remediation model incorporating observ...
2,"RQ4d — Cox model, stratified by year (PH-assum...",0.6138,C-index,—,NaN,Preferred model: retains disclosure year via s...
3,"RQ4e — Cox model, disclosure-year removed (sen...",0.6059,C-index,—,NaN,Secondary sensitivity check assessing whether ...


# RQ4 — Report-ready interpretation

The RQ4 analysis distinguishes between observed remediation duration and time-to-remediation in the presence of right-censored observations.

The OLS analysis is treated as a sensitivity analysis and is restricted to the 61 material-weakness cases for which remediation was observed. This restriction is necessary because the remediation duration of the 52 right-censored cases does not represent an observed completed remediation time.

The Cox proportional hazards model is treated as the primary inferential model because it incorporates all 113 observations while explicitly accounting for the 52 right-censored cases.

The unstratified Cox model evaluates whether weakness group, industry group, and disclosure year are associated with time-to-remediation, but disclosure_year_c fails the proportional-hazards check (p=0.034). RQ4d resolves this by stratifying on binned disclosure year rather than dropping the variable, which is the preferred model going forward; RQ4e (dropping disclosure year entirely) is retained only as a secondary sensitivity check.

### Preferred wording

> The primary time-to-event analysis used a Cox proportional hazards model incorporating all 113 material-weakness observations, including 52 right-censored cases. Because disclosure year violated the proportional-hazards assumption in the unstratified model, the preferred specification stratifies on binned disclosure year (2017-2019, 2020-2021, 2022-2024), which resolves the diagnostic violation while retaining the variable's information. An OLS model restricted to the 61 remediated cases was used as a sensitivity analysis for observed remediation duration. The analysis evaluated weakness category, industry group, and disclosure year as predictors of remediation timing.

> Results should be interpreted as statistical associations rather than causal effects. The absence of statistical significance does not demonstrate that weakness characteristics or industry have no practical relationship with remediation timing; rather, it indicates that the available sample does not provide sufficient evidence for the specified association.

### Do not write

- Weakness category causes slower remediation.
- Industry causes remediation delays.
- The Cox model proves that companies remediate faster.
- A non-significant result proves there is no relationship.

### Preferred terminology

- time-to-remediation
- observed remediation duration
- right-censored observations
- Cox proportional hazards model
- OLS sensitivity analysis
- hazard ratio
- statistical association
- temporal sensitivity
- does not establish causality

In [14]:
# ---------- Save corrected RQ4 results ----------
results = {
    'n_total': int(n_total),
    'n_remediated': int(n_remediated),
    'n_censored': int(n_censored),
    'ols_n': int(reg_model.nobs),
    'ols_r_squared': float(reg_model.rsquared),
    'ols_adjusted_r_squared': float(reg_model.rsquared_adj),
    'ols_f_pvalue': float(reg_model.f_pvalue),
    'cox_concordance_with_year_unstratified': float(cph.concordance_index_),
    'cox_concordance_stratified_by_year': float(cph_strat.concordance_index_),
    'cox_concordance_without_year': float(cph_no_year.concordance_index_),
    'cox_significant_terms_unstratified': {k: float(v) for k, v in cph.summary['p'].items() if v < 0.05},
    'cox_significant_terms_stratified': {k: float(v) for k, v in cph_strat.summary['p'].items() if v < 0.05}
}

with open('rq4_real_results_corrected.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print('\nSaved: rq4_real_results_corrected.json')

{
  "n_total": 113,
  "n_remediated": 61,
  "n_censored": 52,
  "ols_n": 61,
  "ols_r_squared": 0.20015495914630588,
  "ols_adjusted_r_squared": 0.020597909158741823,
  "ols_f_pvalue": 0.3704873716227998,
  "cox_concordance_with_year_unstratified": 0.6199230440359128,
  "cox_concordance_stratified_by_year": 0.6138211382113821,
  "cox_concordance_without_year": 0.6059213339033775,
  "cox_significant_terms_unstratified": {},
  "cox_significant_terms_stratified": {}
}

Saved: rq4_real_results_corrected.json
